# Data Pipeline — AI Model Mumtaz (versi Colab)

Notebook ini menyiapkan **data training** untuk model validasi absen tutor Bimbel Mumtaz.
Isinya sama persis dengan skrip di folder `data_pipeline/` (kode disalin dari file `.py`),
hanya dipecah per bagian dan diberi penjelasan.

Acuan: `docs/IMPLEMENTASI AI MODEL + COMPARISON.md` (repo `mumtaz-bimbel-dashboard`, branch `feat/ui-ux-polish`).

## Alur

```
data_ai.csv (dari tim)
   │  Langkah 1  fix_data_ai          → data_ai_clean.csv            format dirapikan, 8 field
   │  Langkah 2  inject_anomalies     → sessions_with_synthetic.csv  + baris anomali sintetis
   │  Langkah 3  build_training_set   → training_dataset.csv          19 fitur + label
   │                                  → sample_requests.json          contoh request FastAPI
   └─ features.py dipakai di Langkah 3 (dan nantinya di FastAPI)
```

## Cara pakai

1. Di Colab: **Runtime → Run all** (atau jalankan sel satu per satu dari atas).
2. Saat diminta, **upload `data_ai.csv`**.
3. Di bagian akhir, file hasil otomatis bisa di-*download*.

## ⚠️ Privasi data

Data berisi catatan les anak-anak yang asli. **Jangan simpan notebook ini ke GitHub dalam keadaan
ada output** (tabel pratinjau berisi data). Sebelum commit: **Edit → Clear all outputs**.
Simpan file data di Google Drive tim, bukan di repo.

## 0. Persiapan

Semua kode hanya memakai *library* bawaan Python. `pandas` dipakai **hanya untuk menampilkan
pratinjau tabel** di notebook ini (sudah tersedia di Colab).

In [ ]:
import csv
import datetime as dt
import hashlib
import json
import random
import sys
import uuid
from bisect import bisect_left, bisect_right
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)

### Upload data

Sel di bawah meminta file `data_ai.csv`. Di luar Colab (misalnya Jupyter di laptop), sel ini
memakai file `data_ai.csv` di folder yang sama dengan notebook.

> Alternatif lewat Google Drive: `from google.colab import drive; drive.mount('/content/drive')`
> lalu ubah `RAW_PATH` menjadi lokasi file di Drive.

In [ ]:
WORK = Path("pipeline_output")
WORK.mkdir(exist_ok=True)

try:
    from google.colab import files  # hanya ada di Colab
    uploaded = files.upload()
    RAW_PATH = Path(next(iter(uploaded)))
except ImportError:
    RAW_PATH = Path("data_ai.csv")

print("File data:", RAW_PATH, "-", RAW_PATH.stat().st_size // 1024, "KB")
pd.read_csv(RAW_PATH, encoding="utf-8-sig").head()

## Langkah 1 — Merapikan format (`fix_data_ai.py`)

Data dari tim sudah setengah bersih, tetapi formatnya belum sesuai **section 3.2** dokumen.
Langkah ini **hanya memperbaiki format, tidak ada baris yang dihapus**:

| Kolom | Sebelum | Sesudah |
|---|---|---|
| `lesson_date` | `6/16/2025` (bulan/hari/tahun) | `2025-06-16` |
| `start_time` | `9:00` | `09:00` |
| `subject_ids` | urutan acak | diurutkan |
| kolom tambahan | `durationOutlierMinutes`, `lateStartAfter`, 3 kolom kosong | dibuang |

Kasus seperti duplikat atau jam mulai aneh **bukan** kesalahan format — kasus itu diberi label
di Langkah 3. File *issues* hanya berisi masalah format yang sebenarnya.

In [ ]:
FIELDS = ["id", "tutor_id", "student_id", "subject_ids", "lesson_date", "start_time",
          "duration_minutes", "submitted_at"]


MAX_SUBJECTS = 5


def fix_data_ai(src, out_csv, out_issues):
    rows = list(csv.reader(open(src, encoding="utf-8-sig", newline="")))
    if rows[0][:8] != FIELDS:
        sys.exit(f"Unexpected header: {rows[0][:8]}")
    clean, issues = [], []

    def flag(ln, rid, kind, detail):
        issues.append({"csv_line": ln, "id": rid, "issue": kind, "detail": detail})

    for ln, raw in enumerate(rows[1:], start=2):
        if not any(c.strip() for c in raw):
            continue
        r = dict(zip(FIELDS, (c.strip() for c in raw[:8])))
        extra = [c for c in raw[10:] if c.strip()]          # cols 9-10 are config, anything after is stray
        if extra:
            flag(ln, r["id"], "catatan_di_kolom_kosong", f"isi kolom tambahan {extra} (ikut terhapus)")
        try:
            for k in ("id", "tutor_id", "student_id"):
                uuid.UUID(r[k])
            d = dt.datetime.strptime(r["lesson_date"], "%m/%d/%Y").date()   # verified: source is M/D/YYYY
            h, m = map(int, r["start_time"].split(":"))
            assert 0 <= h <= 23 and 0 <= m <= 59
            int(r["duration_minutes"])
            dt.datetime.strptime(r["submitted_at"], "%Y-%m-%dT%H:%M:%SZ")
            subs = json.loads(r["subject_ids"])
            assert isinstance(subs, list) and subs
            for s in subs:
                uuid.UUID(s)
        except Exception as e:   # noqa: BLE001 - report and keep going
            flag(ln, r["id"], "format_tidak_valid", f"{type(e).__name__}: {e}")
            continue
        r["lesson_date"] = d.isoformat()
        r["start_time"] = f"{h:02d}:{m:02d}"
        r["subject_ids"] = json.dumps(sorted(subs))
        if len(subs) > MAX_SUBJECTS:
            flag(ln, r["id"], "mapel_lebih_dari_5", f"{len(subs)} mapel; aplikasi menerima maksimal {MAX_SUBJECTS}")
        if len(set(subs)) != len(subs):
            flag(ln, r["id"], "mapel_berulang", r["subject_ids"])
        clean.append(r)

    ids = Counter(r["id"] for r in clean)
    for r in clean:
        if ids[r["id"]] > 1:
            flag("-", r["id"], "id_ganda", "id sesi muncul lebih dari sekali")

    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows(clean)
    with open(out_issues, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["csv_line", "id", "issue", "detail"])
        w.writeheader()
        w.writerows(issues)
    print(f"{len(clean)} rows written; issues: {dict(Counter(i['issue'] for i in issues)) or 'none'}")

In [ ]:
CLEAN_PATH = WORK / "data_ai_clean.csv"
fix_data_ai(RAW_PATH, CLEAN_PATH, WORK / "data_ai_issues.csv")

display(pd.read_csv(WORK / "data_ai_issues.csv"))
pd.read_csv(CLEAN_PATH).head()

## Aturan bisnis & fitur (`features.py`)

Bagian ini adalah **inti logika**: aturan label dan perhitungan fitur. File `features.py` dibuat
agar bisa dipakai bersama oleh skrip *training* **dan** FastAPI, sehingga fitur saat *training*
dan saat aplikasi berjalan dihitung dengan cara yang sama (section 4.5).

### Aturan label

| Label | Kondisi |
|---|---|
| 🔴 `rejected` | Duplikat persis: tutor, murid, mapel, tanggal, **jam mulai**, dan durasi sama dengan sesi yang di-*submit* lebih dulu |
| 🔵 `needs_revision` | Durasi `> 180` atau `< 30` menit · mulai `> 21:00` atau `< 07:00` · duplikat samar (jam mulai beda) · tutor bentrok dengan murid lain · murid bentrok dengan tutor lain · tutor & murid yang sama bentrok tetapi bukan duplikat persis |
| 🟢 `valid` | Tidak ada temuan (tepat 180/30 menit dan tepat 21:00/07:00 masih valid) |

Prioritas: `rejected` > `needs_revision` > `valid`. Semua temuan tetap dicatat di `findings`.

### Konfigurasi dan daftar fitur

`DEFAULT_CONFIG` berisi batas-batas aturan (bisa diubah tanpa mengubah kode).
`FEATURE_NAMES` adalah **urutan kolom input model** — 13 fitur dari section 3.3 ditambah 6 fitur
baru untuk aturan tambahan. Mengubah urutannya berarti model harus dilatih ulang.

In [ ]:
DEFAULT_CONFIG = {
    "durationOutlierMinutes": 180,   # BLUE when duration > this
    "durationMinMinutes": 30,        # BLUE when duration < this   (AE rule, new)
    "lateStartAfter": "21:00",       # BLUE when start > this
    "earlyStartBefore": "07:00",     # BLUE when start < this      (AE rule, new)
}


# Column order of the model input. Changing it means retraining the model.
FEATURE_NAMES = [
    # A. from target + config (doc 3.3, 1-5)
    "duration_minutes",
    "duration_over_limit",
    "start_minute",
    "start_over_limit",
    "subject_count",
    # B. target vs comparisons (doc 3.3, 6-13)
    "comparisons_count",
    "exact_dup_count",
    "same_tutor_student_date_count",
    "same_student_date_diff_tutor_count",
    "time_overlap_count",
    "tutor_sessions_30d",
    "student_sessions_30d",
    "duration_diff_from_tutor_mean_30d",
    # C. new for the AE's 2026-11-14 rules (14-17)
    "duration_under_limit",
    "start_under_limit",
    "fuzzy_dup_count",
    "tutor_overlap_diff_student_count",
    # D. overlap rules (18-19)
    "student_overlap_diff_tutor_count",
    "self_overlap_count",
]

### Fungsi bantu

- `hhmm_to_minutes("14:00")` → `840` (menit sejak tengah malam), supaya jam bisa dihitung.
- `_subjects(session)` → daftar mapel yang sudah diurutkan, supaya `["B","A"]` dan `["A","B"]` dianggap sama.
- `_overlaps(...)` → `True` bila dua rentang jam **bertabrakan**. Sesi yang bersambung
  (14:00–15:00 lalu 15:00–16:00) **tidak** dianggap bertabrakan.

In [ ]:
def hhmm_to_minutes(value):
    if value is None:
        return None
    h, m = str(value).split(":")
    return int(h) * 60 + int(m)


def _subjects(session):
    return tuple(sorted(session.get("subject_ids") or []))


def _overlaps(a_start, a_dur, b_start, b_dur):
    if a_start is None or b_start is None:
        return False
    return a_start < b_start + b_dur and b_start < a_start + a_dur

### `compute_features(target, comparisons, config)`

Menghitung **19 angka** untuk satu sesi (`target`) dengan membandingkannya ke sesi-sesi
sebelumnya (`comparisons`). Model ML hanya bisa membaca angka, bukan JSON atau UUID.

Cara kerjanya:
1. Untuk setiap sesi pembanding, cek apakah tutor/murid/tanggalnya sama.
2. Hanya sesi **di tanggal yang sama** yang bisa menjadi duplikat atau bentrok jadwal.
3. Hitung jumlah tiap jenis temuan (duplikat persis, duplikat samar, bentrok tutor/murid/sendiri).
4. Fitur "batas" dihitung dengan pengurangan: **nol atau minus = aman, positif = melanggar** (section 3.4).

Fungsi ini juga mengembalikan `related`: ID sesi pembanding yang menyebabkan temuan
(dipakai untuk `related_lesson_session_id` di respons FastAPI).

In [ ]:
def compute_features(target, comparisons, config=None):
    """Return (features: dict in FEATURE_NAMES order, related: dict of finding -> comparison id)."""
    cfg = {**DEFAULT_CONFIG, **(config or {})}
    dur = int(target["duration_minutes"])
    start = hhmm_to_minutes(target.get("start_time"))
    subj = _subjects(target)
    t_id, s_id, date = target["tutor_id"], target["student_id"], target["lesson_date"]

    exact = fuzzy = same_tsd = same_sd_diff_t = overlap = tutor_overlap_diff_s = student_overlap_diff_t = self_overlap = 0
    tutor_n = student_n = 0
    tutor_durations = []
    related = {}

    for c in comparisons:
        c_start = hhmm_to_minutes(c.get("start_time"))
        c_dur = int(c["duration_minutes"])
        c_subj = _subjects(c)
        same_t, same_s, same_d = c["tutor_id"] == t_id, c["student_id"] == s_id, c["lesson_date"] == date

        if same_t:
            tutor_n += 1
            tutor_durations.append(c_dur)
        if same_s:
            student_n += 1
        if not same_d:
            continue

        if same_t and same_s:
            same_tsd += 1
            if c_subj == subj and c_dur == dur:
                if c_start == start:
                    exact += 1
                    related.setdefault("suspected_duplicate", c["id"])
                else:
                    fuzzy += 1
                    related.setdefault("fuzzy_duplicate", c["id"])
        if same_s and not same_t and set(c_subj) & set(subj):
            same_sd_diff_t += 1
        if (same_t or same_s) and _overlaps(start, dur, c_start, c_dur):
            overlap += 1
            if same_t and not same_s:
                tutor_overlap_diff_s += 1
                related.setdefault("tutor_time_overlap", c["id"])
            if same_s and not same_t:
                student_overlap_diff_t += 1
                related.setdefault("student_time_overlap", c["id"])
            is_exact = c_subj == subj and c_dur == dur and c_start == start
            if same_t and same_s and not is_exact:
                self_overlap += 1
                related.setdefault("self_time_overlap", c["id"])

    tutor_mean = sum(tutor_durations) / len(tutor_durations) if tutor_durations else dur
    start_or_zero = start if start is not None else 0
    features = {
        "duration_minutes": dur,
        "duration_over_limit": dur - cfg["durationOutlierMinutes"],
        "start_minute": start_or_zero,
        "start_over_limit": start_or_zero - hhmm_to_minutes(cfg["lateStartAfter"]),
        "subject_count": len(subj),
        "comparisons_count": len(comparisons),
        "exact_dup_count": exact,
        "same_tutor_student_date_count": same_tsd,
        "same_student_date_diff_tutor_count": same_sd_diff_t,
        "time_overlap_count": overlap,
        "tutor_sessions_30d": tutor_n,
        "student_sessions_30d": student_n,
        "duration_diff_from_tutor_mean_30d": round(dur - tutor_mean, 2),
        # Positive = rule broken, 0 or negative = safe (same idea as doc section 3.4)
        "duration_under_limit": cfg["durationMinMinutes"] - dur,
        "start_under_limit": (hhmm_to_minutes(cfg["earlyStartBefore"]) - start) if start is not None else -1,
        "fuzzy_dup_count": fuzzy,
        "tutor_overlap_diff_student_count": tutor_overlap_diff_s,
        "student_overlap_diff_tutor_count": student_overlap_diff_t,
        "self_overlap_count": self_overlap,
    }
    return features, related

### `label_from_features(features)`

Mengubah 19 angka menjadi **label** dan **daftar temuan** sesuai aturan bisnis.
Fungsi ini dipakai untuk memberi label data *training*, dan di FastAPI bisa dipakai sebagai
*guardrail* (pengaman) agar aturan mutlak tidak pernah meleset.

In [ ]:
def label_from_features(f):
    """Rule-based label (the 'guardrail'). Returns (validity, list of finding types)."""
    findings = []
    if f["exact_dup_count"] >= 1:
        findings.append("suspected_duplicate")
    if f["duration_over_limit"] > 0:
        findings.append("duration_outlier")
    if f["duration_under_limit"] > 0:
        findings.append("duration_too_short")
    if f["start_over_limit"] > 0:
        findings.append("late_start")
    if f["start_under_limit"] > 0:
        findings.append("early_start")
    if f["fuzzy_dup_count"] >= 1:
        findings.append("fuzzy_duplicate")
    if f["tutor_overlap_diff_student_count"] >= 1:
        findings.append("tutor_time_overlap")
    if f["student_overlap_diff_tutor_count"] >= 1:
        findings.append("student_time_overlap")
    if f["self_overlap_count"] >= 1:
        findings.append("self_time_overlap")

    if "suspected_duplicate" in findings:
        validity = "rejected"
    elif findings:
        validity = "needs_revision"
    else:
        validity = "valid"
    return validity, findings

### Uji cepat

Memastikan fungsi di atas benar:
- Contoh di **section 4.2** dokumen harus menghasilkan angka yang sama persis.
- Kasus batas: 180 menit valid, 181 tidak; 21:00 valid, 21:01 tidak; dst.
- Contoh dari diskusi: jam mulai sama tetapi durasi beda → `needs_revision`.

Jika ada yang salah, sel ini akan berhenti dengan `AssertionError`.

In [ ]:
T = {"id": "sesi-baru-002", "tutor_id": "tutor-diva", "student_id": "murid-abqary",
     "subject_ids": ["mapel-mtk"], "lesson_date": "2026-09-02", "start_time": "14:00",
     "duration_minutes": 60, "submitted_at": "2026-09-02T08:20:00Z"}
C = [{**T, "id": "sesi-lama-001", "submitted_at": "2026-09-02T08:05:00Z"},
     {**T, "id": "sesi-minggu-lalu-000", "lesson_date": "2026-08-26", "submitted_at": "2026-08-26T09:00:00Z"}]
f, rel = compute_features(T, C)
doc_4_2 = {"duration_minutes": 60, "duration_over_limit": -120, "start_minute": 840, "start_over_limit": -420,
           "subject_count": 1, "comparisons_count": 2, "exact_dup_count": 1, "same_tutor_student_date_count": 1,
           "same_student_date_diff_tutor_count": 0, "time_overlap_count": 1, "tutor_sessions_30d": 2,
           "student_sessions_30d": 2, "duration_diff_from_tutor_mean_30d": 0.0}
assert all(f[k] == v for k, v in doc_4_2.items())
print("Contoh section 4.2 cocok:", label_from_features(f), rel)

def label_of(changes, comparisons=()):
    t = {**T, **changes}
    return label_from_features(compute_features(t, list(comparisons))[0])[0]

cases = [
    ({"duration_minutes": 180}, [], "valid"), ({"duration_minutes": 181}, [], "needs_revision"),
    ({"duration_minutes": 30}, [], "valid"), ({"duration_minutes": 29}, [], "needs_revision"),
    ({"start_time": "21:00"}, [], "valid"), ({"start_time": "21:01"}, [], "needs_revision"),
    ({"start_time": "07:00"}, [], "valid"), ({"start_time": "06:59"}, [], "needs_revision"),
    ({}, [{**T, "id": "x", "duration_minutes": 90}], "needs_revision"),              # jam sama, durasi beda
    ({}, [{**T, "id": "x", "student_id": "lain", "start_time": "14:30"}], "needs_revision"),  # tutor bentrok
    ({}, [{**T, "id": "x", "tutor_id": "lain", "start_time": "14:30"}], "needs_revision"),    # murid bentrok
    ({}, [{**T, "id": "x", "student_id": "lain", "start_time": "15:00"}], "valid"),           # bersambung
    ({}, [{**T, "id": "x"}], "rejected"),                                            # duplikat persis
]
for changes, comps, expected in cases:
    assert label_of(changes, comps) == expected, (changes, expected)
print(f"{len(cases)} kasus uji lolos")

## Fungsi penyusun `comparisons` (`build_training_set.py`)

Fungsi-fungsi ini didefinisikan **sebelum** Langkah 2 karena Langkah 2 memakainya untuk
mengetahui label data asli terlebih dahulu.

### Memuat data

`load_sessions` membaca CSV, `prepare_sessions` mengubah teks menjadi tipe yang tepat
(tanggal, angka, daftar mapel) dan mencatat apakah baris itu sintetis.

In [ ]:
WINDOW_DAYS = 30


MAX_COMPARISONS = 100


SESSION_FIELDS = ["id", "tutor_id", "student_id", "subject_ids", "lesson_date", "start_time",
                  "duration_minutes", "submitted_at"]


def load_sessions(path):
    return prepare_sessions(csv.DictReader(open(path, encoding="utf-8-sig", newline="")))


def prepare_sessions(records):
    sessions = []
    for r in records:
        s = {k: r[k] for k in SESSION_FIELDS}
        s["subject_ids"] = json.loads(s["subject_ids"]) if isinstance(s["subject_ids"], str) else list(s["subject_ids"])
        s["duration_minutes"] = int(s["duration_minutes"])
        s["_date"] = dt.date.fromisoformat(s["lesson_date"])
        s["_synthetic"] = str(r.get("is_synthetic", "0")) in ("1", "true", "True")
        s["_scenario"] = r.get("scenario") or "real"
        sessions.append(s)
    return sessions


def public(s):
    return {k: s[k] for k in SESSION_FIELDS}

### `build_comparisons(target, by_person)`

Menyusun daftar sesi pembanding untuk satu `target`, sama seperti query *backend* saat aplikasi
berjalan (section 3.1). Sebuah sesi masuk `comparisons` bila:

1. **Tutor atau murid sama** dengan `target`,
2. **Tanggal les** berjarak maksimal **±30 hari**,
3. **Di-*submit* lebih dulu** daripada `target` (`submitted_at` lebih kecil).

Maksimal **100 sesi**. Bila lebih dari 100, yang diambil adalah sesi dengan tanggal les
**paling dekat** — jadi sesi di tanggal yang sama (yang menentukan duplikat & bentrok) selalu ikut.

`by_person` adalah indeks sesi per tutor dan per murid yang sudah diurutkan per tanggal, sehingga
pencarian cepat (`bisect`) walau datanya ribuan baris.

**Kenapa `submitted_at` penting?** Absen yang masuk **duluan** dianggap asli (`valid`), yang masuk
**belakangan** dianggap duplikat (`rejected`), karena saat absen pertama masuk, kembarannya belum ada.

In [ ]:
def build_comparisons(target, by_person):
    """Sessions of the same tutor OR student, +-30 days, submitted earlier, max 100."""
    lo = target["_date"] - dt.timedelta(days=WINDOW_DAYS)
    hi = target["_date"] + dt.timedelta(days=WINDOW_DAYS)
    seen, pool = set(), []
    for key in (("t", target["tutor_id"]), ("s", target["student_id"])):
        dates, rows = by_person[key]
        for s in rows[bisect_left(dates, lo):bisect_right(dates, hi)]:
            if s["id"] != target["id"] and s["id"] not in seen and s["submitted_at"] < target["submitted_at"]:
                seen.add(s["id"])
                pool.append(s)
    # If more than 100, keep the sessions closest in lesson_date, newest submissions first.
    # NOTE for the backend: the live query should pick the same 100 (rarely matters; see report).
    pool.sort(key=lambda s: (abs((s["_date"] - target["_date"]).days), _neg(s["submitted_at"])))
    return pool[:MAX_COMPARISONS], len(pool)


def _neg(ts):
    return [-ord(ch) for ch in ts]

### `label_sessions(sessions)`

Untuk **setiap sesi**: susun `comparisons` → hitung 19 fitur → beri label. Hasilnya satu baris per
sesi berisi fitur, label (`validity`), temuan (`findings`), dan ID sesi pembanding yang terkait.

Fungsi ini juga menyimpan **satu contoh request per jenis temuan** (`sample_requests.json`) dalam
format `{target, comparisons, config}` — format yang akan dikirim *backend* ke FastAPI.

In [ ]:
def label_sessions(sessions):
    """Build comparisons, features and labels for every session. Returns (rows, samples, capped)."""
    by_person = defaultdict(lambda: ([], []))
    for s in sorted(sessions, key=lambda s: s["_date"]):
        for key in (("t", s["tutor_id"]), ("s", s["student_id"])):
            by_person[key][0].append(s["_date"])
            by_person[key][1].append(s)

    rows, samples, capped = [], {}, 0
    for s in sessions:
        comps, pool_size = build_comparisons(s, by_person)
        capped += pool_size > MAX_COMPARISONS
        feats, related = compute_features(public(s), [public(c) for c in comps], DEFAULT_CONFIG)
        validity, findings = label_from_features(feats)
        rel = next((related[f] for f in ("suspected_duplicate", "fuzzy_duplicate", "tutor_time_overlap", "student_time_overlap", "self_time_overlap")
                    if f in related), "")
        rows.append({"session_id": s["id"], **feats, "related_session_id": rel,
                     "findings": ";".join(findings), "validity": validity,
                     "scenario": s["_scenario"], "is_synthetic": int(s["_synthetic"])})
        # keep one small example payload per (validity, first finding), preferring few comparisons
        sample_key = f"{validity}:{findings[0] if findings else 'none'}"
        if sample_key not in samples or len(comps) < len(samples[sample_key]["request"]["comparisons"]):
            samples[sample_key] = {"expected_validity": validity, "expected_findings": findings,
                                   "request": {"target": public(s), "comparisons": [public(c) for c in comps],
                                               "config": DEFAULT_CONFIG}}

    return rows, samples, capped

### Label data asli (sebelum ditambah data sintetis)

Kita lihat dulu komposisi label data asli. Biasanya sebagian besar `valid`, dan beberapa aturan
hampir tidak punya contoh — itulah alasan Langkah 2 diperlukan.

In [ ]:
real_rows, _, _ = label_sessions(load_sessions(CLEAN_PATH))
real_df = pd.DataFrame(real_rows)
display(real_df["validity"].value_counts().to_frame("jumlah").assign(
    persen=lambda d: (d["jumlah"] / d["jumlah"].sum() * 100).round(1)))
real_df["findings"].str.split(";").explode().replace("", pd.NA).dropna().value_counts().to_frame("jumlah temuan")

## Langkah 2 — Menambah anomali sintetis (`inject_anomalies.py`)

Model tidak bisa belajar aturan yang hampir tidak pernah ia lihat. Karena itu ditambahkan
**baris anomali buatan** (section 5.5) sampai komposisi akhir **60% valid / 20% needs_revision /
20% rejected**.

### Pengaturan

- `TARGET_SHARE`: komposisi label akhir yang diinginkan.
- `FIXED_COUNTS`: skenario dengan jumlah tetap (contoh batas yang valid, dan pasangan "kombinasi").
- `NEEDS_REVISION_SCENARIOS`: skenario `needs_revision`, jumlahnya dibagi rata.
- `plan_counts()`: **menghitung otomatis** berapa baris sintetis per skenario berdasarkan label data
  asli, sehingga komposisi tetap tepat walaupun data asli berubah.

| Skenario | Isi |
|---|---|
| `durasi_panjang` / `durasi_pendek` | durasi 181–300 / 10–29 menit |
| `mulai_malam` / `mulai_pagi` | mulai 21:01–22:30 / 05:00–06:59 |
| `duplikat_samar` | salinan sesi asli dengan jam mulai berbeda |
| `tumpang_tindih` / `_murid` / `_sendiri` | bentrok jadwal tutor / murid / tutor+murid yang sama |
| `duplikat` | salinan persis sesi asli, di-*submit* belakangan → `rejected` |
| `kombinasi_dup_*` | sesi tidak wajar + salinan persisnya (dua temuan sekaligus) |
| `batas_valid` | tepat 180/30 menit, tepat 21:00/07:00 → `valid` (mengajari model letak batas) |

In [ ]:
SEED = 20261114


FIELDS = ["id", "tutor_id", "student_id", "subject_ids", "lesson_date", "start_time",
          "duration_minutes", "submitted_at"]


DAY_START, DAY_END = 7 * 60, 21 * 60


LAST_MINUTE = 23 * 60 + 59


# Final share of each label in the training data (real + synthetic).
TARGET_SHARE = {"valid": 0.60, "needs_revision": 0.20, "rejected": 0.20}


# Scenarios with a fixed count. A "kombinasi" pair adds 2 rows per count:
# an unusual session (needs_revision) + its exact copy (rejected, both findings).
FIXED_COUNTS = {
    "batas_valid": 100,           # exactly 180/30 min, exactly 21:00/07:00 -> valid (teaches the edges)
    "kombinasi_dup_panjang": 50,  # long session + copy
    "kombinasi_dup_pendek": 50,   # short session + copy
    "kombinasi_dup_malam": 50,    # late session + copy
    "kombinasi_dup_pagi": 50,     # early session + copy
}


# The rest is calculated in plan_counts():
#   needs_revision, split evenly over these 7 scenarios
NEEDS_REVISION_SCENARIOS = [
    "durasi_panjang",        # 181-300 min
    "durasi_pendek",         # 10-29 min
    "mulai_malam",           # start 21:01-22:30
    "mulai_pagi",            # start 05:00-06:59
    "duplikat_samar",        # copy of a real session, different start_time
    "tumpang_tindih",        # same tutor, other student, overlapping time
    "tumpang_tindih_murid",  # same student, other tutor, overlapping time
    "tumpang_tindih_sendiri",  # same tutor + student, overlapping, not exact copy
]


def plan_counts(real_labels):
    """How many synthetic rows per scenario so the final data matches TARGET_SHARE."""
    pairs = sum(v for k, v in FIXED_COUNTS.items() if k.startswith("kombinasi"))
    valid_total = real_labels["valid"] + FIXED_COUNTS["batas_valid"]
    total = round(valid_total / TARGET_SHARE["valid"])
    want_nr = round(total * TARGET_SHARE["needs_revision"])
    want_rej = total - valid_total - want_nr
    nr_singles = want_nr - real_labels["needs_revision"] - pairs
    dup = want_rej - real_labels["rejected"] - pairs
    if nr_singles < 0 or dup < 0:
        sys.exit("TARGET_SHARE cannot be reached by adding rows; lower FIXED_COUNTS or change the shares")
    counts = dict(FIXED_COUNTS, duplikat=dup)
    per, extra = divmod(nr_singles, len(NEEDS_REVISION_SCENARIOS))
    for i, name in enumerate(NEEDS_REVISION_SCENARIOS):
        counts[name] = per + (1 if i < extra else 0)
    return counts


def hm(minutes):
    return f"{minutes // 60:02d}:{minutes % 60:02d}"


def ts(value):
    return dt.datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ")


def ts_str(value):
    return value.strftime("%Y-%m-%dT%H:%M:%SZ")

### Kelas `Injector`

Membuat baris sintetis dengan **aman**:

- Setiap baris sintetis dibuat dari **sesi asli yang valid** (tutor, murid, mapel asli), lalu diubah
  sesuai skenario.
- Baris ditempatkan di **hari atau slot jam yang kosong**, dan waktu *submit* diatur, sehingga
  **label baris asli tidak ikut berubah**.
- Setiap baris diberi ID baru, `is_synthetic=1`, dan nama `scenario`.
- Memakai *seed* tetap (`SEED`), jadi hasilnya **selalu sama** setiap dijalankan.

Metode penting:
- `standalone()` — satu sesi tidak wajar (durasi/jam) di hari yang kosong.
- `exact_copy()` / `fuzzy_copy()` — duplikat persis / duplikat samar.
- `overlap_other_student()` / `overlap_other_tutor()` / `self_overlap()` — tiga jenis bentrok jadwal.
- `run()` — menjalankan semua skenario sesuai jumlah dari `plan_counts()`.

In [ ]:
class Injector:
    def __init__(self, real_rows, rnd, counts):
        self.rnd = rnd
        self.counts = counts
        self.self_overlap_variant = 0
        self.rows = real_rows
        self.synthetic = []
        self.day = defaultdict(list)          # (kind, person, date) -> [(start, end)]
        self.students_of_tutor = defaultdict(set)
        self.tutors_of_student = defaultdict(set)
        self.all_tutors = sorted({r["tutor_id"] for r in real_rows})
        self.all_subjects = sorted({sid for r in real_rows for sid in json.loads(r["subject_ids"])})
        self.max_submit = defaultdict(lambda: dt.datetime.min)   # (kind, person) -> latest submitted_at
        for r in real_rows:
            self._occupy(r)
            self.students_of_tutor[r["tutor_id"]].add(r["student_id"])
            self.tutors_of_student[r["student_id"]].add(r["tutor_id"])
        # base pool: real sessions that are clean & unambiguous
        self.base_pool = self._clean_bases()

    # ---------------------------------------------------------------- bookkeeping
    def _slot(self, r):
        s = hhmm_to_minutes(r["start_time"])
        return s, s + int(r["duration_minutes"])

    def _occupy(self, r):
        for key in (("t", r["tutor_id"]), ("s", r["student_id"])):
            self.day[(*key, r["lesson_date"])].append(self._slot(r))
            self.max_submit[key] = max(self.max_submit[key], ts(r["submitted_at"]))

    def _free(self, kind, person, date, start, end, ignore=None):
        for a, b in self.day[(kind, person, date)]:
            if (a, b) == ignore:
                ignore = None          # skip exactly one matching slot (the base session)
                continue
            if start < b and a < end:
                return False
        return True

    def _clean_bases(self):
        count = Counter((r["tutor_id"], r["student_id"], r["lesson_date"]) for r in self.rows)
        pool = []
        for r in self.rows:
            s, e = self._slot(r)
            if (count[(r["tutor_id"], r["student_id"], r["lesson_date"])] == 1
                    and DAY_START <= s and e <= DAY_END and 30 <= int(r["duration_minutes"]) <= 180
                    and self._free("t", r["tutor_id"], r["lesson_date"], s, e, ignore=(s, e))
                    and self._free("s", r["student_id"], r["lesson_date"], s, e, ignore=(s, e))):
                pool.append(r)
        return pool

    def _new_id(self):
        return str(uuid.UUID(int=self.rnd.getrandbits(128), version=4))

    def _add(self, row, scenario):
        row = {**row, "id": self._new_id(), "is_synthetic": 1, "scenario": scenario}
        self.synthetic.append(row)
        self._occupy(row)
        return row

    def _submit_after_lesson(self, row, latest_than=None):
        """Lesson end (WIB) + 5..180 min, and later than `latest_than` if given."""
        end_local = dt.datetime.fromisoformat(row["lesson_date"]) + dt.timedelta(minutes=self._slot(row)[1])
        t = end_local - dt.timedelta(hours=7) + dt.timedelta(minutes=self.rnd.randint(5, 180),
                                                               seconds=self.rnd.randint(0, 59))
        if latest_than and t <= latest_than:
            t = latest_than + dt.timedelta(minutes=self.rnd.randint(1, 600), seconds=self.rnd.randint(0, 59))
        return ts_str(t)

    def _free_day_for(self, base, start, end, max_tries=60):
        """A date near the base where neither tutor nor student has any session."""
        d0 = dt.date.fromisoformat(base["lesson_date"])
        for _ in range(max_tries):
            d = (d0 + dt.timedelta(days=self.rnd.choice([-1, 1]) * self.rnd.randint(1, 20))).isoformat()
            if not self.day[("t", base["tutor_id"], d)] and not self.day[("s", base["student_id"], d)]:
                return d
        return None

    # ---------------------------------------------------------------- scenarios
    def standalone(self, scenario, start, duration):
        """One unusual session on a day the tutor and student are otherwise free.
        Submitted after every real session of that tutor/student, so no real row
        ever has it in its comparisons."""
        for _ in range(200):
            base = self.rnd.choice(self.base_pool)
            date = self._free_day_for(base, start, start + duration)
            if not date:
                continue
            row = {**base, "lesson_date": date, "start_time": hm(start), "duration_minutes": duration}
            latest = max(self.max_submit[("t", base["tutor_id"])], self.max_submit[("s", base["student_id"])])
            row["submitted_at"] = self._submit_after_lesson(row, latest_than=latest)
            return self._add(row, scenario)
        raise RuntimeError(f"no free day found for {scenario}")

    def exact_copy(self, base, scenario):
        row = {k: base[k] for k in FIELDS}
        row["submitted_at"] = ts_str(ts(base["submitted_at"]) + dt.timedelta(
            minutes=self.rnd.randint(1, 4320), seconds=self.rnd.randint(0, 59)))
        return self._add(row, scenario)

    def fuzzy_copy(self, base):
        s, e = self._slot(base)
        dur = e - s
        slots = [m for m in range(DAY_START, DAY_END - dur + 1, 15)
                 if m != s and self._free("t", base["tutor_id"], base["lesson_date"], m, m + dur)
                 and self._free("s", base["student_id"], base["lesson_date"], m, m + dur)]
        if not slots:
            return None
        row = {k: base[k] for k in FIELDS}
        row["start_time"] = hm(self.rnd.choice(slots))
        row["submitted_at"] = ts_str(ts(base["submitted_at"]) + dt.timedelta(
            minutes=self.rnd.randint(1, 4320), seconds=self.rnd.randint(0, 59)))
        return self._add(row, "duplikat_samar")

    def overlap_other_student(self, base):
        s, e = self._slot(base)
        others = [st for st in self.students_of_tutor[base["tutor_id"]]
                  if st != base["student_id"] and not self.day[("s", st, base["lesson_date"])]]
        if not others:
            return None
        dur = self.rnd.choice([60, 60, 60, 90, 45])
        start = self.rnd.randint(max(DAY_START, s - dur + 15), min(e - 15, DAY_END - dur))
        start -= start % 5
        if not (start < e and s < start + dur):
            return None
        # must overlap ONLY the base session of this tutor
        if not self._free("t", base["tutor_id"], base["lesson_date"], start, start + dur, ignore=(s, e)):
            return None
        row = {k: base[k] for k in FIELDS}
        row.update(student_id=self.rnd.choice(sorted(others)), start_time=hm(start), duration_minutes=dur)
        row["submitted_at"] = ts_str(ts(base["submitted_at"]) + dt.timedelta(
            minutes=self.rnd.randint(1, 4320), seconds=self.rnd.randint(0, 59)))
        return self._add(row, "tumpang_tindih")

    def overlap_other_tutor(self, base):
        s, e = self._slot(base)
        date = base["lesson_date"]
        known = [t for t in self.tutors_of_student[base["student_id"]]
                 if t != base["tutor_id"] and not self.day[("t", t, date)]]
        others = known or [t for t in self.all_tutors if t != base["tutor_id"] and not self.day[("t", t, date)]]
        if not others:
            return None
        dur = self.rnd.choice([60, 60, 60, 90, 45])
        start = self.rnd.randint(max(DAY_START, s - dur + 15), min(e - 15, DAY_END - dur))
        start -= start % 5
        if not (start < e and s < start + dur):
            return None
        # must overlap ONLY the base session of this student
        if not self._free("s", base["student_id"], date, start, start + dur, ignore=(s, e)):
            return None
        row = {k: base[k] for k in FIELDS}
        row.update(tutor_id=self.rnd.choice(sorted(others)), start_time=hm(start), duration_minutes=dur)
        row["submitted_at"] = ts_str(ts(base["submitted_at"]) + dt.timedelta(
            minutes=self.rnd.randint(1, 4320), seconds=self.rnd.randint(0, 59)))
        return self._add(row, "tumpang_tindih_murid")

    def self_overlap(self, base):
        """Same tutor + same student + same date, overlapping the base, but NOT an exact
        copy and NOT a fuzzy duplicate (so the only finding is self_time_overlap).
        Three variants, in rotation:
          0 same start, different duration       (e.g. 14:00 60 min vs 14:00 90 min)
          1 same start + duration, different subjects
          2 start shifted 5-30 min AND different duration"""
        s, e = self._slot(base)
        dur = e - s
        variant = self.self_overlap_variant % 3
        row = {k: base[k] for k in FIELDS}
        if variant == 0:
            new_dur = self.rnd.choice([d for d in range(30, 181, 5) if d != dur])
            start = s
        elif variant == 1:
            subs = json.loads(base["subject_ids"])
            others = [x for x in self.all_subjects if x not in subs]
            if len(subs) > 1 and self.rnd.random() < 0.5:
                new_subs = subs[:-1]                                  # one subject fewer
            else:
                new_subs = subs[:-1] + [self.rnd.choice(others)]      # one subject swapped
            row["subject_ids"] = json.dumps(sorted(new_subs))
            start, new_dur = s, dur
        else:
            start = s + self.rnd.choice([-1, 1]) * self.rnd.randint(1, 6) * 5
            new_dur = self.rnd.choice([d for d in range(30, 181, 5) if d != dur])
        end = start + new_dur
        if not (DAY_START <= start and end <= DAY_END and start < e and s < end):
            return None
        # overlap ONLY the base, for both tutor and student
        if not (self._free("t", base["tutor_id"], base["lesson_date"], start, end, ignore=(s, e))
                and self._free("s", base["student_id"], base["lesson_date"], start, end, ignore=(s, e))):
            return None
        row.update(start_time=hm(start), duration_minutes=new_dur)
        row["submitted_at"] = ts_str(ts(base["submitted_at"]) + dt.timedelta(
            minutes=self.rnd.randint(1, 4320), seconds=self.rnd.randint(0, 59)))
        self.self_overlap_variant += 1
        return self._add(row, f"tumpang_tindih_sendiri")

    def run(self):
        rnd = self.rnd
        cfg = DEFAULT_CONFIG
        max_dur, min_dur = cfg["durationOutlierMinutes"], cfg["durationMinMinutes"]
        late, early = hhmm_to_minutes(cfg["lateStartAfter"]), hhmm_to_minutes(cfg["earlyStartBefore"])

        def long_():
            dur = rnd.randint(max_dur + 1, 300)
            return rnd.choice(range(DAY_START, LAST_MINUTE - dur, 15)), dur

        def short():
            return rnd.choice(range(13 * 60, 19 * 60, 15)), rnd.randint(10, min_dur - 1)

        def late_():
            start = rnd.randint(late + 1, 22 * 60 + 30)
            return start, rnd.randint(min_dur, min(120, LAST_MINUTE - start))

        def early_():
            return rnd.randint(5 * 60, early - 1), rnd.choice([60, 60, 90, 45, 120])

        shapes = {"panjang": long_, "pendek": short, "malam": late_, "pagi": early_}
        for name, shape in (("durasi_panjang", long_), ("durasi_pendek", short),
                            ("mulai_malam", late_), ("mulai_pagi", early_)):
            for _ in range(self.counts[name]):
                self.standalone(name, *shape())
        for kind, shape in shapes.items():
            name = f"kombinasi_dup_{kind}"
            for _ in range(self.counts[name]):
                self.exact_copy(self.standalone(name, *shape()), name)

        edges = [(13 * 60, max_dur), (14 * 60, min_dur), (late, 60), (early, 60)]
        for i in range(self.counts["batas_valid"]):
            start, dur = edges[i % 4]
            self.standalone("batas_valid", start, dur)

        # copies of real sessions: each base used once
        bases = self.base_pool[:]
        rnd.shuffle(bases)
        used = 0
        for scenario, fn in (("duplikat", lambda b: self.exact_copy(b, "duplikat")),
                             ("duplikat_samar", self.fuzzy_copy),
                             ("tumpang_tindih", self.overlap_other_student),
                             ("tumpang_tindih_murid", self.overlap_other_tutor),
                             ("tumpang_tindih_sendiri", self.self_overlap)):
            made = 0
            while made < self.counts[scenario] and used < len(bases):
                base = bases[used]
                used += 1
                if fn(base):
                    made += 1
            if made < self.counts[scenario]:
                sys.exit(f"Not enough clean real sessions: only {made}/{self.counts[scenario]} rows for "
                         f"{scenario}. Lower TARGET_SHARE for that label.")
        return self.synthetic

### Menjalankan Langkah 2

`inject_anomalies()` memberi label data asli, menghitung rencana jumlah per skenario, membuat baris
sintetis, lalu menyimpan gabungan data asli + sintetis.

In [ ]:
def inject_anomalies(src, dst):
    rnd = random.Random(SEED)
    real = []
    for r in csv.DictReader(open(src, encoding="utf-8-sig", newline="")):
        real.append({**{k: r[k] for k in FIELDS}, "duration_minutes": int(r["duration_minutes"]),
                     "is_synthetic": 0, "scenario": "real"})
    real_rows, _, _ = label_sessions(prepare_sessions(real))
    real_labels = Counter(r["validity"] for r in real_rows)
    counts = plan_counts(real_labels)
    print("real labels:", dict(real_labels))
    print("planned synthetic rows:", counts)
    synthetic = Injector(real, rnd, counts).run()
    rows = real + synthetic
    for r in rows:
        r["subject_ids"] = r["subject_ids"] if isinstance(r["subject_ids"], str) else json.dumps(r["subject_ids"])
    with open(dst, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS + ["is_synthetic", "scenario"])
        w.writeheader()
        w.writerows(rows)
    print(f"real {len(real)} + synthetic {len(synthetic)} = {len(rows)} rows")
    print("synthetic per scenario:", dict(Counter(r["scenario"] for r in synthetic)))

In [ ]:
SYN_PATH = WORK / "sessions_with_synthetic.csv"
inject_anomalies(CLEAN_PATH, SYN_PATH)
pd.read_csv(SYN_PATH)["scenario"].value_counts().to_frame("jumlah baris")

## Langkah 3 — Menyusun data *training* (`build_training_set.py`)

Menjalankan `label_sessions` pada data gabungan (asli + sintetis), lalu menyimpan:

- `training/training_dataset.csv` — satu baris per sesi: 19 fitur, `validity`, `findings`,
  `related_session_id`, `scenario`, `is_synthetic`. **Siap untuk `model.fit(X, y)`**.
- `training/sample_requests.json` — contoh request untuk menguji FastAPI.

In [ ]:
def build_training_set(src, out_dir):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    rows, samples, capped = label_sessions(load_sessions(src))
    cols = ["session_id", *FEATURE_NAMES, "related_session_id", "findings", "validity", "scenario", "is_synthetic"]
    with open(out / "training_dataset.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        w.writerows(rows)
    (out / "sample_requests.json").write_text(json.dumps(dict(sorted(samples.items())), indent=2, ensure_ascii=False))

    n = len(rows)
    print(f"sessions: {n}  (real {n - sum(r['is_synthetic'] for r in rows)}, synthetic {sum(r['is_synthetic'] for r in rows)})")
    print(f"targets with more than {MAX_COMPARISONS} candidate comparisons: {capped}")
    print("comparisons per target: max", max(r["comparisons_count"] for r in rows),
          " mean", round(sum(r["comparisons_count"] for r in rows) / n, 1))
    for label, c in Counter(r["validity"] for r in rows).most_common():
        print(f"  {label:15s} {c:5d}  ({c / n:.1%})")
    print("findings:", dict(Counter(f for r in rows for f in r["findings"].split(";") if f).most_common()))
    print("label by source:", dict(Counter((r["is_synthetic"], r["validity"]) for r in rows)))

In [ ]:
TRAIN_DIR = WORK / "training"
build_training_set(SYN_PATH, TRAIN_DIR)
train = pd.read_csv(TRAIN_DIR / "training_dataset.csv")
train.head()

### Hasil: komposisi label

Tabel pertama: komposisi akhir (harus 60 / 20 / 20).
Tabel kedua: label per sumber data — terlihat berapa yang asli dan berapa yang sintetis.

In [ ]:
display(train["validity"].value_counts().to_frame("jumlah").assign(
    persen=lambda d: (d["jumlah"] / len(train) * 100).round(1)))
pd.crosstab(train["validity"], train["is_synthetic"].map({0: "asli", 1: "sintetis"}), margins=True)

### Hasil: jenis temuan

Setiap aturan sekarang punya cukup contoh untuk dipelajari model.

In [ ]:
train["findings"].str.split(";").explode().replace("", pd.NA).dropna().value_counts().to_frame("jumlah")

## Pemeriksaan

Dua pemeriksaan penting:

1. **Label asli tidak berubah** — label setiap baris asli harus sama dengan label sebelum data sintetis
   ditambahkan (Langkah 2 tidak boleh "mengotori" data asli).
2. **Cek ulang aturan secara terpisah** — label dihitung ulang dengan kode sederhana yang **tidak memakai**
   `features.py`, lalu dibandingkan. Hasil harus 0 perbedaan.

In [ ]:
# 1. label baris asli sebelum vs sesudah data sintetis
before = real_df.set_index("session_id")[["validity", "findings"]]
after = train[train["is_synthetic"] == 0].set_index("session_id")[["validity", "findings"]].fillna("")
changed = (before.fillna("") != after.loc[before.index]).any(axis=1).sum()
print("Baris asli yang labelnya berubah:", changed)

# 2. cek ulang aturan dengan kode terpisah (brute force per tanggal)
sessions = list(csv.DictReader(open(SYN_PATH, encoding="utf-8")))
mins = lambda t: int(t[:2]) * 60 + int(t[3:])
for s in sessions:
    s["st"], s["du"], s["sub"] = mins(s["start_time"]), int(s["duration_minutes"]), tuple(sorted(json.loads(s["subject_ids"])))
by_date = defaultdict(list)
for s in sessions:
    by_date[s["lesson_date"]].append(s)
expected_label = {}
for t in sessions:
    earlier = [c for c in by_date[t["lesson_date"]] if c is not t and c["submitted_at"] < t["submitted_at"]]
    overlap = lambda c: t["st"] < c["st"] + c["du"] and c["st"] < t["st"] + t["du"]
    same5 = lambda c: c["tutor_id"] == t["tutor_id"] and c["student_id"] == t["student_id"] and c["sub"] == t["sub"] and c["du"] == t["du"]
    exact = any(same5(c) and c["st"] == t["st"] for c in earlier)
    blue = (t["du"] > 180 or t["du"] < 30 or t["st"] > 21 * 60 or t["st"] < 7 * 60
            or any(same5(c) and c["st"] != t["st"] for c in earlier)
            or any(c["tutor_id"] == t["tutor_id"] and c["student_id"] != t["student_id"] and overlap(c) for c in earlier)
            or any(c["student_id"] == t["student_id"] and c["tutor_id"] != t["tutor_id"] and overlap(c) for c in earlier)
            or any(c["tutor_id"] == t["tutor_id"] and c["student_id"] == t["student_id"] and overlap(c)
                   and not (same5(c) and c["st"] == t["st"]) for c in earlier))
    expected_label[t["id"]] = "rejected" if exact else "needs_revision" if blue else "valid"
mismatch = (train.set_index("session_id")["validity"] != pd.Series(expected_label)).sum()
print("Label yang berbeda dari cek ulang:", mismatch, "dari", len(train))
assert changed == 0 and mismatch == 0

## Download hasil

Mengunduh file hasil ke komputer. Simpan di **Google Drive tim**, jangan di GitHub.

In [ ]:
outputs = [CLEAN_PATH, SYN_PATH, TRAIN_DIR / "training_dataset.csv", TRAIN_DIR / "sample_requests.json"]
try:
    from google.colab import files
    for p in outputs:
        files.download(str(p))
except ImportError:
    for p in outputs:
        print("Tersimpan:", p)

## Langkah berikutnya (AI Engineer)

- Kolom fitur untuk model: `FEATURE_NAMES` (19 kolom), target: `validity`.
- Sisihkan sebagian **baris asli** (`is_synthetic == 0`) sebagai data uji agar akurasi model
  diukur pada data nyata, bukan data buatan.
- Gunakan `features.py` di FastAPI agar fitur saat *live* dihitung sama persis dengan saat *training*.

⚠️ Sebelum menyimpan notebook ini ke GitHub: **Edit → Clear all outputs**.